In [1]:
from simbanator.io.simba import Simulation
from simbanator.analysis.particles import extract_particles
from astropy.io import fits
import numpy as np
# Load your simulation (use the name you configured)
sim = Simulation("cis100")  # Replace XX with your snapshot number

[Simulation] Loading snap→z mapping from: /mnt/home/glorenzon/analize_simba_cgm/simbanator/data/snap_z_maps/zsnap_map_caesar_box100.txt


In [2]:
from pathlib import Path
import textwrap
import subprocess

targetfile = '/mnt/home/glorenzon/analize_simba_cgm/output/cis100/quenching_times/forward_modeled_unique_sample.fits'
with fits.open(targetfile) as hdul:
    data = hdul[1].data
    id_draws = np.asarray(data['GROUPID_SNAPSHOT'], dtype=np.int64)
    snaps = np.asarray(data['SNAPSHOT'], dtype=np.int64)
    print(snaps)

# unique (snapshot, galaxy_id) pairs -> one array task per pair
jobs = np.unique(np.column_stack([snaps, id_draws]), axis=0)
if jobs.size == 0:
    raise RuntimeError('No jobs found in FITS table.')

job_dir = Path.cwd() / 'output' / 'slurm' / 'filter_particles'
log_dir = job_dir / 'logs'
job_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

manifest_path = job_dir / 'jobs.tsv'
np.savetxt(
    manifest_path,
    jobs,
    fmt='%d\t%d',
    header='snap\tgalaxy_id',
    comments=''
 )

worker_path = job_dir / 'run_filter_particles_worker.py'
worker_path.write_text(
    textwrap.dedent(
        '''
        import argparse
        from pathlib import Path
        import numpy as np
        from simbanator.io.simba import Simulation
        from simbanator.analysis.particles import extract_particles

        def main():
            parser = argparse.ArgumentParser()
            parser.add_argument('--manifest', required=True)
            parser.add_argument('--task-id', type=int, required=True)
            parser.add_argument('--simulation', default='cis100')
            args = parser.parse_args()

            rows = np.loadtxt(args.manifest, dtype=np.int64, skiprows=1)
            if rows.ndim == 1:
                rows = rows[np.newaxis, :]

            if args.task_id < 0 or args.task_id >= len(rows):
                raise IndexError(
                    f'task_id={args.task_id} is out of range for {len(rows)} jobs'
                )

            snap, galaxy_id = rows[args.task_id]
            sim = Simulation(args.simulation)
            obj = sim.load_catalog(int(snap))
            snap_path = sim.get_snapshot_file(int(snap))

            snap_dir = Path.cwd() / 'output' / args.simulation / 'filtered' / f'snap_{int(snap):03d}'
            snap_dir.mkdir(parents=True, exist_ok=True)

            output_file = snap_dir / f'snap_m100n1024_{int(snap):03d}_snap{int(snap)}_gal{int(galaxy_id)}.h5'

            extract_particles(
                obj,
                snap_path,
                snap=int(snap),
                galaxy_id=int(galaxy_id),
                output=str(output_file),
                sim_name=sim.name
            )
            print(f'Completed snap={int(snap)} galaxy_id={int(galaxy_id)} -> {output_file}')

        if __name__ == '__main__':
            main()
        '''
    ).strip() + '\n'
 )

slurm_path = job_dir / 'submit_filter_particles_array.sh'
slurm_path.write_text(
    textwrap.dedent(
        f'''
        #!/bin/bash
        #SBATCH --job-name=flt_parts
        #SBATCH --output={log_dir}/%x_%A_%a.out
        #SBATCH --error={log_dir}/%x_%A_%a.err
        #SBATCH --time=04:00:00
        #SBATCH --cpus-per-task=1
        #SBATCH --mem=8G
        #SBATCH --array=0-{len(jobs)-1}

        set -euo pipefail
        cd {Path.cwd()}
        source $HOME/miniconda3/etc/profile.d/conda.sh
        conda activate pd39
        python {worker_path} --manifest {manifest_path} --task-id ${{SLURM_ARRAY_TASK_ID}}
        '''
    ).strip() + '\n'
 )
slurm_path.chmod(0o750)

submit_jobs = False  # flip to True to submit now
if submit_jobs:
    res = subprocess.run(['sbatch', str(slurm_path)], check=True, text=True, capture_output=True)
    print(res.stdout.strip())
else:
    print(f'Prepared {len(jobs)} array tasks.')
    print(f'Manifest: {manifest_path}')
    print(f'SLURM script: {slurm_path}')
    print(f'To submit: sbatch {slurm_path}')
    print('Outputs will be written under output/<simulation>/filtered/snap_XXX/')

[124 123 122 121 121 120 120 120 120 120 120 120 120 120 120 120 120 120
 120 120 120 120 120 120 120 120 120 120 120 120 120 119 119 119 119 119
 119 119 119 119 119 119 119 119 119 119 119 118 118 118 118 118 118 118
 118 118 118 118 118 118 118 118 118 118 118 118 118 118 118 118 117 117
 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117
 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117
 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117
 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 117 116 116
 116 116 116 116 116 116 116 116 116 116 116 116 116 116 116 116 116 116
 116 116 116 116 116 116 116 116 116 116 116 116 116 116 116 116 116 116
 116 116 116 116 116 116 116 116 116 116 116 116 116 116 115 115 115 115
 115 115 115 115 115 115 115 115 115 115 115 115 115 115 115 115 115 115
 115 115 115 115 115 115 115 115 115 115 115 115 115 115 115 115 115 115
 115 114 114 114 114 114 114 114 114 114 114 114 11

In [7]:
from collections import defaultdict
import numpy as np

targetfile = '/mnt/home/glorenzon/analize_simba_cgm/output/cis100/quenching_times/forward_modeled_unique_sample.fits'
sim = Simulation('cis100')
with fits.open(targetfile) as hdul:
    data = hdul[1].data
    id_draws = np.asarray(data['GROUPID_SNAPSHOT'], dtype=np.int64)
    snaps = np.asarray(data['SNAPSHOT'], dtype=np.int64)

# --- Input arrays ---
snaps = np.array(snaps)
ids = np.array(id_draws)

# --- 1. Group galaxy IDs by snapshot ---
snap_dict = defaultdict(list)
for snap, gid in zip(snaps, ids):
    snap_dict[int(snap)].append(int(gid))

# --- 2. Sort snapshots ---
sorted_snaps = sorted(snap_dict.keys())

# --- 3. Optional: chunking function ---
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

# --- 4. Main loop ---
for snap in sorted_snaps:
    galaxy_ids = snap_dict[snap]

    print(f"\nProcessing snapshot {snap} with {len(galaxy_ids)} galaxies")

    # Load catalog and snapshot ONCE per snapshot
    obj = sim.load_catalog(snap)
    snap_path = sim.get_snapshot_file(snap)
    if snap in [92, 94, 96, 98, 99]:
        continue
    extract_particles(
        obj,
        snap_path,
        snap=snap,
        galaxy_ids=galaxy_ids,   # list of galaxies
        sim_name=sim.name,
        overwrite=False,
        prefix='m100n1024'
    )